<a href="https://colab.research.google.com/github/nuhuynhh/IT-Support-AI-Agent/blob/main/Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Install libraries

In [60]:
!pip install langchain langchain-openai langchain-community langgraph chromadb pypdf langchain-chroma langchain-text-splitters
import os
from typing import List, Dict, Optional, Literal, Any
from datetime import datetime

import chromadb
from pydantic import BaseModel, Field, ConfigDict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# simple sample with LangChain + LangGraph, OpenAI for LLM + embeddings, “knowledge base” is just a few uploaded PDFs (policies, runbooks, etc.), 4 agents: Intake, Knowledge (RAG), Workflow, Escalation
# use of chroma https://docs.trychroma.com/docs/overview/getting-started and public mcp server https://www.mcpserverfinder.com/servers/sirmews/mcp-pinecone

# step1 setup and ingest PDFs

Step 2: API keys/imports

In [61]:
import os
from google.colab import userdata

# It's recommended to store API keys securely, e.g., using Colab's Secrets feature.
# For demonstration, you can uncomment the line below and replace with your actual key.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# If using Colab Secrets, you can retrieve it like this:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Step 3: Build PDF vector database

In [62]:
from typing import List
import os

def build_pdf_vectorstore(pdf_paths: List[str], persist_dir: str = "./chroma_pdf_kb") -> Chroma:
    docs = []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
    )

    for path in pdf_paths:
        loader = PyPDFLoader(path)
        pages = loader.load()

        for page in pages:
            page.metadata["source"] = "pdf"
            page.metadata["file_name"] = os.path.basename(path)

        docs.extend(pages)

    split_docs = text_splitter.split_documents(docs)

    vectorstore = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        persist_directory=persist_dir,
    )

    return vectorstore


pdf_paths = [
    "/content/Corporate VPN Access Policy.pdf",
    "/content/Password Reset Runbook.pdf",
    "/content/IT Onboarding Guide for New Employees.pdf",
]

for path in pdf_paths:
    print(path, os.path.exists(path))

chroma = build_pdf_vectorstore(pdf_paths)
kb_retriever = chroma.as_retriever(search_kwargs={"k": 5})

/content/Corporate VPN Access Policy.pdf True
/content/Password Reset Runbook.pdf True
/content/IT Onboarding Guide for New Employees.pdf True


Test Retrieval:

In [63]:
docs = kb_retriever.invoke("How do I reset my password?")

for doc in docs:
    print(doc.metadata)
    print(doc.page_content[:300])
    print("-----")

{'source': 'pdf', 'page_label': '1', 'page': 0, 'creationdate': '', 'creator': 'PyPDF', 'title': 'Password Reset Runbook', 'file_name': 'Password Reset Runbook.pdf', 'producer': 'Skia/PDF m149 Google Docs Renderer', 'total_pages': 2}
When  to  Use  This  Runbook  
●  User  forgot  password  ●  Account  is  locked  ●  Password  expired  
 
Self-Service  Password  Reset  Steps  
1.  Go  to  the  password  reset  portal  2.  Enter  your  company  email  or  username  3.  Verify  identity  using  MFA  4.  Create  a  new  password  f
-----
{'source': 'pdf', 'file_name': 'Password Reset Runbook.pdf', 'creationdate': '', 'producer': 'Skia/PDF m149 Google Docs Renderer', 'title': 'Password Reset Runbook', 'creator': 'PyPDF', 'page_label': '1', 'total_pages': 2, 'page': 0}
When  to  Use  This  Runbook  
●  User  forgot  password  ●  Account  is  locked  ●  Password  expired  
 
Self-Service  Password  Reset  Steps  
1.  Go  to  the  password  reset  portal  2.  Enter  your  company  email  or  

Step 4: Define GraphState models

# Pydantic GraphState, GraphState.retrieved_docs will hold chunks coming from PDFs.

In [64]:
# Pydantic GraphState, GraphState.retrieved_docs will hold chunks coming from PDFs.
#NOTES FOR OURSELVES DELETE LATER?
# Graphstate = shared memory of our system
# packages the info that moves between each step
# just defines the structure of the data

from typing import List, Dict, Optional, Literal
from pydantic import BaseModel, Field
from datetime import datetime

#
class RetrievedDoc(BaseModel):
    id: str                         # unique ID of the chunk
    source: str                     # "confluence", "kb", "itsm", etc. (where it came from; pdf)
    title: str                      # doc name
    snippet: str                    # The text chunk from the PDF
    score: float                    # How relevant it is to the query
    url: Optional[str] = None
    metadata: Dict[str, str] = Field(default_factory=dict)


class WorkflowAction(BaseModel):    # an action the system wants to take
    action_type: Literal[           # menu of possible actions (toolbox)
        "RESET_PASSWORD",
        "UNLOCK_ACCOUNT",
        "GET_LOGS",
        "OPEN_TICKET",
        "CUSTOM"
    ]
    system: Optional[str] = None     # "VPN", "Email", etc. (what the system applies to)
    parameters: Dict[str, str] = Field(default_factory=dict)    # inputs like user_id
    status: Literal["PENDING", "RUNNING", "SUCCESS", "FAILED"] = "PENDING"  # tracks progress
    result_summary: Optional[str] = None    # what happened after the action
    error_message: Optional[str] = None


class ToolResult(BaseModel):         # what comes back from the MCP tools
    tool_name: str                   # what tool was used
    success: bool                    # determines if it works/doesn't
    payload: Dict = Field(default_factory=dict)   # extra data(like ticket ID, logs)
    error: Optional[str] = None
    timestamp: datetime = Field(default_factory=datetime.utcnow)


class Intent(BaseModel):
    name: Optional[str] = None       # what intake agent figures out: "reset_password", "vpn_issue", etc.
    confidence: float = 0.0
    request_type: Literal[           # type of request
        "informational",
        "action",
        "incident",
        "unknown"
    ] = "unknown"
    system: Optional[str] = None     # what the system relates to (VPN, Email, Account)
    severity: Optional[Literal["low", "medium", "high"]] = None  # Intent = what the user wants + how serious


class GraphState(BaseModel):      # Entire system's memory
    # conversation messages for LangGraph state (history)
    messages: List[Dict] = Field(default_factory=list)
    # each item: {"role": "user"/"assistant"/"system", "content": "..."}

    # user/session (user info)
    user_id: Optional[str] = None
    user_department: Optional[str] = None
    user_roles: List[str] = Field(default_factory=list)

    # intake  (output of the Intake Agent)
    intent: Intent = Field(default_factory=Intent)
    entities: Dict[str, str] = Field(default_factory=dict)    # What controls routing

    # RAG     (Knowledge Agent)
    rag_query: Optional[str] = None     # password reset steps
    retrieved_docs: List[RetrievedDoc] = Field(default_factory=list)    # gets neceesary docs determined by intake agent(intent)
    knowledge_answer: Optional[str] = None

    # Workflow Agent (where the system does the work)
    workflow_actions: List[WorkflowAction] = Field(default_factory=list)
    tool_results: List[ToolResult] = Field(default_factory=list) # where MCP gets used

    # Escalation Agent (where the system can't solve the problem)
    escalation_required: bool = False
    escalation_reason: Optional[str] = None
    escalation_ticket_id: Optional[str] = None
    escalation_summary: Optional[str] = None    # where MCP -> Github comes in

    # final user-facing answer
    final_answer: Optional[str] = None


Mock MCP Client

In [65]:
# STEP 5: Mock MCP Client
class MockMCPClient:
    def call(self, tool_name, **kwargs):
        if tool_name == "idp.reset_password":
            return {
                "summary": f"Password reset simulated for user {kwargs.get('user_id')}.",
                "status": "success"
            }

        if tool_name == "logs.get_recent":
            return {
                "summary": "Logs checked. VPN authentication failure detected.",
                "logs": ["Failed VPN login", "MFA timeout", "User retry detected"]
            }

        if tool_name in ["itsm.create_ticket", "itsm.create_incident"]:
            return {
                "ticket_id": "IT-1001",
                "summary": "Support ticket created successfully."
            }

        return {"error": "Unknown tool"}

mcp_client = MockMCPClient()

Step 5: Intake Agent

# Intake Agent Node


In [66]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, Literal

class IntakeSchema(BaseModel):
    model_config = ConfigDict(extra="forbid")

    intent_name: Optional[str] = Field(
        default=None,
        description="Intent name such as reset_password, unlock_account, vpn_issue, onboarding_help, or unknown"
    )
    confidence: float = Field(default=0.0)
    request_type: Literal["informational", "action", "incident", "unknown"] = "unknown"
    system: Optional[str] = None
    severity: Optional[Literal["low", "medium", "high"]] = None

def intake_node(state: GraphState) -> GraphState:
    last_user_msg = next(
        (m for m in reversed(state.messages) if m["role"] == "user"),
        None,
    )
    if not last_user_msg:
        return state

    prompt = f"""
You are an IT support intake agent.

Task:
1. Determine the intent name (e.g. "reset_password", "vpn_issue", "how_to_request_access").
2. Determine request_type: ["informational", "action", "incident", "unknown"].
3. Determine system (e.g. "VPN", "Email", "SSO", "Laptop", "Unknown").
4. Extract entities like username, device, application, times, etc.
5. Estimate severity: "low", "medium", "high" (or "unknown").
6. Provide a confidence (0-1).

User message:
{last_user_msg["content"]}
"""

    structured_llm = llm.with_structured_output(
    IntakeSchema,
    method="function_calling"
)
    result: IntakeSchema = structured_llm.invoke(prompt)

    state.intent = Intent(
        name=result.intent_name,
        confidence=result.confidence,
        request_type=result.request_type,
        system=None if result.system.lower() == "unknown" else result.system,
        severity=None if result.severity == "unknown" else result.severity,
    )
    state.entities = {}
    return state

Step 6: Knowledge Agent

# Knowledge Agent Node, RAG over PDFs


In [67]:
# knowledge agent node, RAG over PDFs

knowledge_prompt = ChatPromptTemplate.from_template("""
You are an IT knowledge agent answering user questions using internal PDF documentation.

User question:
{question}

Context from documents:
{context}

Instructions:
- Answer based ONLY on the provided context.
- If multiple procedures exist, pick the one most relevant to the user's department and system.
- If you are not sure, say so and suggest next steps.
- Provide a clear, concise answer; include a short numbered procedure if appropriate.
""")


def knowledge_node(state: GraphState) -> GraphState:
    # Only run for informational/incident types
    if state.intent.request_type not in ["informational", "incident"]:
        return state

    last_user_msg = next(
        (m for m in reversed(state.messages) if m["role"] == "user"),
        None,
    )
    if not last_user_msg:
        return state

    system_part = f" (system: {state.intent.system})" if state.intent.system else ""
    rag_query = last_user_msg["content"] + system_part
    state.rag_query = rag_query

    # ---- Retrieve from your uploaded PDFs ----
    docs = kb_retriever.invoke(rag_query)

    state.retrieved_docs = []
    for i, d in enumerate(docs):
        meta = d.metadata or {}

        # Convert all metadata values to strings
        safe_meta = {k: str(v) for k, v in meta.items()}

        state.retrieved_docs.append(
            RetrievedDoc(
                id=str(meta.get("id", f"pdf-{i}")),
                source=str(meta.get("source", "pdf")),
                title=str(meta.get("file_name", "PDF Document")),
                snippet=d.page_content[:400],
                score=float(meta.get("score", 0.0)),
                url=None,
                metadata=safe_meta,
            )
        )

    context_text = "\n\n".join(
        f"[Doc {i+1} - {doc.title}]\n{doc.snippet}"
        for i, doc in enumerate(state.retrieved_docs)
    )

    chain = knowledge_prompt | llm
    answer_msg = chain.invoke(
        {"question": last_user_msg["content"], "context": context_text}
    )

    state.knowledge_answer = answer_msg.content

    # If purely informational, we can answer directly
    if state.intent.request_type == "informational":
        state.final_answer = state.knowledge_answer

    return state

Step 6: Knowledge Agent

# Workflow Agent Node, LangChain + MCP tools over Pinecone/IDP/ITSM

In [68]:
# workflow agent node, LangChain + MCP tools over Pinecone/IDP/ITSM
# helper
def plan_workflow_actions(state: GraphState) -> List[WorkflowAction]:
    intent = state.intent

    if intent.request_type == "action" and intent.name in ["reset_password", "unlock_account"]:
        return [
            WorkflowAction(
                action_type="RESET_PASSWORD",
                system=intent.system or "Unknown",
                parameters={
                    "user_id": state.user_id,
                    "reason": "Forgot password / account locked"
                },
            ),
            WorkflowAction(
                action_type="OPEN_TICKET",
                system=intent.system or "Unknown",
                parameters={
                    "reason": "Password reset performed automatically",
                    "description": "Automated password reset and account unlock initiated by AI agent."
                },
            ),
        ]

    if intent.request_type == "incident":
        return [
            WorkflowAction(
                action_type="GET_LOGS",
                system=intent.system or "Unknown",
                parameters={
                    "service": intent.system or "VPN",
                    "timeframe": "24h",
                    "user_id": state.user_id,
                },
            )
        ]

    return []

# implementation

def workflow_node(state: GraphState) -> GraphState:
    """
    LangGraph node: Workflow Agent
    Uses mcp_client to call tools (idp.reset_password, logs.get_recent, itsm.create_ticket, etc.).
    """

    # Plan only once
    if not state.workflow_actions:
        state.workflow_actions = plan_workflow_actions(state)

    for action in state.workflow_actions:
        if action.status not in ["PENDING", "RUNNING"]:
            continue

        action.status = "RUNNING"
        tool_name = "unknown"

        try:
            if action.action_type == "RESET_PASSWORD":
                tool_name = "idp.reset_password"
                payload = mcp_client.call(
                    tool_name,
                    user_id=state.user_id, # Use state.user_id from GraphState
                    system=action.system,
                )

            elif action.action_type == "GET_LOGS":
                tool_name = "logs.get_recent"
                payload = mcp_client.call(
                    tool_name,
                    service=action.parameters["service"],
                    timeframe=action.parameters["timeframe"],
                    user_id=state.user_id, # Use state.user_id from GraphState
                )

            elif action.action_type == "OPEN_TICKET":
                tool_name = "itsm.create_ticket"
                payload = mcp_client.call(
                    tool_name,
                    summary=f"Auto action for {action.system}",
                    description=action.parameters.get("reason", "Automated action"),
                    user_id=state.user_id, # Use state.user_id from GraphState
                    severity=state.intent.severity or "low",
                )

            else:
                payload = {"error": f"Unsupported action_type: {action.action_type}"}
                raise ValueError(payload["error"])

            success = not payload.get("error")
            action.status = "SUCCESS" if success else "FAILED"
            action.result_summary = payload.get("summary") or str(payload)

            state.tool_results.append(
                ToolResult(
                    tool_name=tool_name,
                    success=success,
                    payload=payload,
                    error=payload.get("error"),
                )
            )

            if tool_name.startswith("itsm.") and payload.get("ticket_id"):
                state.escalation_ticket_id = payload["ticket_id"]

        except Exception as e:
            action.status = "FAILED"
            action.error_message = str(e)
            state.tool_results.append(
                ToolResult(
                    tool_name=tool_name,
                    success=False,
                    payload={},
                    error=str(e),
                )
            )

    # If all actions succeeded → build final_answer
    if state.workflow_actions and all(a.status == "SUCCESS" for a in state.workflow_actions):
        summary_lines = [
            f"- {a.action_type} on {a.system}: {a.result_summary}"
            for a in state.workflow_actions
        ]
        ticket_part = (
            f"\nRelated ticket: {state.escalation_ticket_id}"
            if state.escalation_ticket_id
            else ""
        )
        state.final_answer = (
            "I've completed the requested actions:\n" +
            "\n".join(summary_lines) +
            ticket_part
        )
    # If any failed, mark for escalation
    elif any(a.status == "FAILED" for a in state.workflow_actions):
        state.escalation_required = True
        state.escalation_reason = "One or more workflow actions failed."

    return state

Step 8: Escalation Agent

# Escalation Agent Node, LangChain + MCP ITSM tool

In [69]:
# Escalation Agent Node, LangChain + MCP ITSM tool

from langchain_core.prompts import ChatPromptTemplate

escalation_prompt = ChatPromptTemplate.from_template("""
You are an IT escalation agent.

Summarize the situation for a human IT engineer. Include:
- User problem description
- Steps already taken (by user and by automation)
- Any diagnostic data or logs that seem relevant
- Your best hypothesis if applicable
- Suggested next steps for the human engineer.

Conversation (last turns):
{conversation}

Tool results:
{tool_results}
""")

def escalation_node(state: GraphState) -> GraphState:
    """LangGraph node: Escalation Agent."""

    should_escalate = (
        state.escalation_required
        or (state.intent.severity == "high")
        or (state.intent.request_type == "incident" and state.intent.confidence < 0.6)
    )

    if not should_escalate:
        return state

    convo_text = "\n".join(
        f"{m['role']}: {m['content']}"
        for m in state.messages[-10:]
    )

    tool_summaries = "\n".join(
        f"{i+1}. {tr.tool_name} (success={tr.success}) -> {tr.payload}"
        for i, tr in enumerate(state.tool_results)
    )

    chain = escalation_prompt | llm
    summary_msg = chain.invoke(
        {"conversation": convo_text, "tool_results": tool_summaries}
    )

    summary = summary_msg.content
    state.escalation_summary = summary

    payload = mcp_client.call(
        "itsm.create_incident",
        summary=f"AI Escalation: {state.intent.name or 'IT Issue'}",
        description=summary,
        user_id=state.user_id,
        severity=state.intent.severity or "medium",
    )

    ticket_id = payload.get("incident_id") or payload.get("ticket_id")
    state.escalation_ticket_id = ticket_id

    state.tool_results.append(
        ToolResult(
            tool_name="itsm.create_incident",
            success=not payload.get("error"),
            payload=payload,
            error=payload.get("error"),
        )
    )

    state.final_answer = (
        f"I’ve escalated this issue to our IT team for further investigation.\n\n"
        f"Reference: {ticket_id or 'pending'}\n\n"
        "They’ll review the diagnostics I’ve attached and follow up with you."
    )
    state.escalation_required = True

    return state

# Langgraph Orchestration


Step 9: LangGraph routing

In [70]:
# LangGraph orchestration

from langgraph.graph import StateGraph, END

graph = StateGraph(GraphState)

# Add agent nodes
graph.add_node("intake", intake_node)
graph.add_node("knowledge", knowledge_node)
graph.add_node("workflow", workflow_node)
graph.add_node("escalation", escalation_node)

# Start with Intake Agent
graph.set_entry_point("intake")


# Route after Intake Agent
def route_from_intake(state: GraphState) -> str:
    if state.intent.request_type in ["informational", "incident"]:
        return "knowledge"
    if state.intent.request_type == "action":
        return "workflow"
    return "escalation"


# Route after Knowledge Agent
def route_after_knowledge(state: GraphState) -> str:
    if not state.knowledge_answer:
        return "escalation"
    if state.intent.request_type == "incident":
        return "workflow"
    return END


# Route after Workflow Agent
def route_after_workflow(state: GraphState) -> str:
    if state.escalation_required:
        return "escalation"
    return END


# Add conditional routing
graph.add_conditional_edges(
    "intake",
    route_from_intake,
    {
        "knowledge": "knowledge",
        "workflow": "workflow",
        "escalation": "escalation",
    }
)

graph.add_conditional_edges(
    "knowledge",
    route_after_knowledge,
    {
        "workflow": "workflow",
        "escalation": "escalation",
        END: END,
    }
)

graph.add_conditional_edges(
    "workflow",
    route_after_workflow,
    {
        "escalation": "escalation",
        END: END,
    }
)

graph.add_edge("escalation", END)

app = graph.compile()

In [71]:
test_cases = [
    "I forgot my password and my account is locked.",
    "My VPN is not connecting and I keep getting authentication errors.",
    "I am a new employee. How do I set up my account, MFA, and VPN?"
]

for query in test_cases:
    print("=" * 80)
    print("USER:", query)

    test_state = GraphState(
        messages=[{"role": "user", "content": query}],
        user_id="nathan123",
        user_department="Business",
        user_roles=["student"]
    )

    result = app.invoke(test_state)

    print("INTENT:", result["intent"])
    print("FINAL ANSWER:", result["final_answer"])
    print("RETRIEVED DOCS:", [doc.title for doc in result["retrieved_docs"]])
    print("WORKFLOW ACTIONS:", result["workflow_actions"])
    print("TOOL RESULTS:", result["tool_results"])

USER: I forgot my password and my account is locked.
INTENT: name='reset_password' confidence=0.9 request_type='action' system=None severity='high'
FINAL ANSWER: I've completed the requested actions:
- RESET_PASSWORD on Unknown: Password reset simulated for user nathan123.
- OPEN_TICKET on Unknown: Support ticket created successfully.
Related ticket: IT-1001
RETRIEVED DOCS: []
WORKFLOW ACTIONS: [WorkflowAction(action_type='RESET_PASSWORD', system='Unknown', parameters={'user_id': 'nathan123', 'reason': 'Forgot password / account locked'}, status='SUCCESS', result_summary='Password reset simulated for user nathan123.', error_message=None), WorkflowAction(action_type='OPEN_TICKET', system='Unknown', parameters={'reason': 'Password reset performed automatically', 'description': 'Automated password reset and account unlock initiated by AI agent.'}, status='SUCCESS', result_summary='Support ticket created successfully.', error_message=None)]
TOOL RESULTS: [ToolResult(tool_name='idp.reset_pa